---

## 1. Neo4J Desktop 환경 설정

- **다운로드 및 설치**: https://neo4j.com/deployment-center/?desktop-gdb
    - Neo4J 5.24.0 선택
    - 새 프로젝트 생성 및 DBMS 추가

- **APOC 플러그인 설정**: 
    - APOC 플러그인을 설치하려는 데이터베이스가 있는 프로젝트(Graph DBMS)를 선택
    - Graph DBMS 메뉴 클릭하고, APOC 플러그인(Plugin) 설치

- **설정 파일 수정**: 데이터베이스를 중지한 상태에서 데이터베이스 카드의 오른쪽에 있는 `...` (메뉴) 버튼을 클릭

    - 메뉴에서 **Settings** 선택하고 다음 내용을 추가 (`neo4j.conf` 파일)
        ```
        dbms.security.procedures.unrestricted=apoc.meta.*,apoc.*
        ```
- **nori 형태소 분석기 설치**: 
    - 메뉴에서 **Open foler** 선택하고 다음 파일을 이동하여 저장
        - `neo4j-nori-analyzer-5.24.0.jar` 파일을 Neo4j의 `plugins` 폴더에 복사
    - Neo4J 브라우저 도구를 실행(Open 버튼 클릭)하고 다음 쿼리를 실행하고 'nori' 토크나이저를 목록에서 확인
        ```cypher
        CALL db.index.fulltext.listAvailableAnalyzers()
        ```

In [1]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

In [2]:
from langchain_neo4j import Neo4jGraph

# Neo4j Desktop 연결 설정
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True
)

c:\Users\yoonj\vscodeProject\GraphRAG-w.neo4J-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 테스트 쿼리 실행 
cypher_query = """
MATCH (n) 
RETURN count(n) AS node_count
"""

graph.query(cypher_query)

[{'node_count': 0}]

---

## 2. Docling 한국어 문서 처리

-  **법률문서 PDF 문서 구조를 추출, 변환**
- 주택임대차보호법, 시행령, 시행규칙 (출처: https://www.law.go.kr/)

* **Docling 설치 방법**

    ```python
    pip install docling
    ```

In [4]:
from glob import glob

#법령 파일 경로 리스트 확인
law_files = glob("law_data/*.pdf")
law_files

['law_data\\근로기준법 시행규칙(고용노동부령)(제00436호)(20250223).pdf',
 'law_data\\근로기준법 시행령(대통령령)(제35276호)(20250223).pdf',
 'law_data\\근로기준법(법률)(제20520호)(20250223).pdf']

In [64]:
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from pathlib import Path
from tqdm import tqdm
import json

# OCR 설정
pipeline_options = PdfPipelineOptions()
pipeline_options.ocr_options = EasyOcrOptions(lang=["ko"])  # 한국어 OCR 설정

# DocumentConverter 인스턴스 생성
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# 각 파일별로 변환 및 처리
for law_file in tqdm(law_files, desc="법률 문서 처리 중"):
    file_path = Path(law_file)
    print(f"\n파일 '{file_path.name}' 변환 중...")
    
    try:
        # 문서 변환
        result = converter.convert(file_path)

        # 변환 성공 확인
        if result.status == "success":
            # 문서 정보 출력
            print(f"  - 페이지 수: {len(result.document.pages)}")
            
            # 마크다운으로 내용 추출 (처음 500자만 예시로 표시)
            markdown_content = result.document.export_to_markdown()
            print(f"  - 내용 미리보기: {markdown_content[:500]}...")
            
            # 결과를 마크다운 파일로 저장 
            output_path = Path("law_data") / f"processed_{file_path.stem}.md"
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(markdown_content)
            print(f"  - 변환된 내용이 {output_path}에 저장되었습니다.")

        else:
            print(f"  - 변환 실패: {result.status}")
    except Exception as e:
        print(f"  - 오류 발생: {str(e)}")
    
    print("-" * 50)

법률 문서 처리 중:   0%|          | 0/3 [00:00<?, ?it/s]


파일 '근로기준법 시행규칙(고용노동부령)(제00436호)(20250223).pdf' 변환 중...


법률 문서 처리 중:  33%|███▎      | 1/3 [01:02<02:05, 62.99s/it]

  - 페이지 수: 4
  - 내용 미리보기: ## 근로기준법 시행규칙

[시행 2025. 2. 23.] [고용노동부령 제436호, 2025. 2. 21., 일부개정]

고용노동부 (임금근로시간정책과 - 근로시간, 휴게) 044-202-7545

고용노동부 (근로기준정책과 - 해고, 취업규칙, 기타) 044-202-7534

고용노동부 (근로기준정책과 - 임금) 044-202-7548

고용노동부 (여성고용정책과 - 여성) 044-202-7475

고용노동부 (근로기준정책과 - 소년) 044-202-7535

고용노동부 (임금근로시간정책과 - 제63조 적용제외, 특례업종) 044-202-7530

고용노동부 (임금근로시간정책과 - 휴일, 연차휴가) 044-202-7973

고용노동부 (임금근로시간정책과 - 유연근로시간제) 044-202-7549

- 제1조(목적) 이 규칙은 「근로기준법」과 같은 법 시행령에서 위임한 사항과 그 시행에 필요한 사항을 규정하는 것을 목 적으로 한다.
- 제2조(손해배상 청구의 신청) 근로자는 「근로...
  - 변환된 내용이 law_data\processed_근로기준법 시행규칙(고용노동부령)(제00436호)(20250223).md에 저장되었습니다.
--------------------------------------------------

파일 '근로기준법 시행령(대통령령)(제35276호)(20250223).pdf' 변환 중...


법률 문서 처리 중:  67%|██████▋   | 2/3 [01:16<00:33, 33.99s/it]

  - 페이지 수: 13
  - 내용 미리보기: ## 근로기준법 시행령

[시행 2025. 2. 23.] [대통령령 제35276호, 2025. 2. 18., 일부개정]

고용노동부 (임금근로시간정책과 - 근로시간, 휴게) 044-202-7545

고용노동부 (근로기준정책과 - 소년) 044-202-7535

고용노동부 (근로기준정책과) 044-202-7546

고용노동부 (근로기준정책과 - 임금) 044-202-7548

고용노동부 (여성고용정책과 - 여성) 044-202-7475

고용노동부 (근로기준정책과 - 해고, 취업규칙, 기타) 044-202-7534

고용노동부 (임금근로시간정책과 - 제63조 적용제외, 특례업종) 044-202-7530

고용노동부 (임금근로시간정책과 - 휴일, 연차휴가) 044-202-7973

고용노동부 (임금근로시간정책과 - 유연근로시간제) 044-202-7549

제1조(목적) 이 영은 「근로기준법」에서 위임한 사항과 그 시행에 필요한 사항을 규정하는 것을 목적으로 한다.

제2조(평균임금의 계...
  - 변환된 내용이 law_data\processed_근로기준법 시행령(대통령령)(제35276호)(20250223).md에 저장되었습니다.
--------------------------------------------------

파일 '근로기준법(법률)(제20520호)(20250223).pdf' 변환 중...


법률 문서 처리 중: 100%|██████████| 3/3 [01:37<00:00, 32.52s/it]

  - 페이지 수: 21
  - 내용 미리보기: ## 제1장 총칙

- 제1조(목적) 이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.

제2조(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. &lt;개정 2018. 3. 20., 2019. 1. 15., 2020. 5. 26.&gt;

1. '근로자'란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. '사용자'란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를 말한다.
3. '근로'란 정신노동과 육체노동을 말한다.
4. '근로계약'이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체 결된 계약을 말한다.
5. '임금'이란 사용자가 근로의 대가로 근로자에게 임금, 봉급, 그 밖에 어떠한 명칭으로든지 지급하는 모든 금품을 말한다.
6. '...
  - 변환된 내용이 law_data\processed_근로기준법(법률)(제20520호)(20250223).md에 저장되었습니다.
--------------------------------------------------


---

## 3. **Knowledge Graph 구축**

### 3.1 데이터 로드


In [5]:
#마크다운 데이터 로드
import glob
from pathlib import Path

#처리된 마크다운 파일 목록 가져오기
processed_md_files = glob.glob("law_data/processed_*.md")

#각 파일의 내용 로드
law_contents = {}
for md_file in processed_md_files:
    file_path = Path(md_file)
    law_name = file_path.stem.replace("processed_","")

    with open(file_path,"r", encoding="utf-8") as f:
        content = f.read()
    
    law_contents[law_name] = content

print(f"로드된 법률 문서: {list(law_contents.keys())}")

로드된 법률 문서: ['근로기준법 시행규칙(고용노동부령)(제00436호)(20250223)', '근로기준법 시행령(대통령령)(제35276호)(20250223)', '근로기준법(법률)(제20520호)(20250223)']


### 3.2 각 법률 문서를 그래프로 변환

- **LaborLawKGExtractor** 클래스는 근로기준법 관련 문서에서 **지식 그래프**를 추출함
- 코드는 법률 문서에서 **장, 조, 항, 호** 등의 계층적 구조를 추출
- Neo4j 데이터베이스에 **노드와 관계**를 생성하여 법률 온톨로지를 구축
- 추출된 구조는 **GraphDocument** 객체로 변환되어 Neo4j에 저장


In [6]:
import re
from langchain_neo4j import Neo4jGraph
import os
from langchain_neo4j.graphs.graph_document import GraphDocument,Node, Relationship
from tqdm import tqdm

class LaborLawKGExtractor:
    """근로기준법 관련 문서에서 지식 그래프를 추출하는 클래스"""
    def __init__(self, law_contents):
        """
        초기화 함수

        Args:
            law_contents (dict): 법률 문서 내용을 담은 딕셔너리
        """
        self.law_contents = law_contents
        self.node_dict = {}
        self.relationships =[]

        # Neo4j 데이터베이스 연결 설정
        self.graph = Neo4jGraph(
            url=os.getenv("NEO4J_URI"),
            username=os.getenv("NEO4J_USERNAME"),
            password=os.getenv("NEO4J_PASSWORD"),
            database=os.getenv("NEO4J_DATABASE"),
            enhanced_schema=True,
            refresh_schema=True  
        )
    
    def extract_structure(self):
        """
        문서에서 장, 조, 항, 호 등의 계층적 구조를 추출
        
        Returns:
            dict: 법률별로 계층적 구조를 가진 딕셔너리
        """
        all_laws = {}
        
        # 각 법률 문서에 대해 처리
        for law_name, content in self.law_contents.items():
            # 문서 구조 추출
            law_structure = self.extract_sections(content, law_name)
            all_laws[law_name] = law_structure
        
        return all_laws
    
    def extract_sections(self, content, law_name):
        """
        문서에서 섹션(장, 조 등)을 계층적으로 추출
        
        Args:
            content (str): 법률 문서 내용
            law_name (str): 법률 문서 이름
            
        Returns:
            dict: 계층적 구조를 가진 법률 문서 딕셔너리
        """
        law_structure = {
            'name': law_name,  # 법률 이름
            'type': 'law',  # 노드 타입
            'full_text': content,  # 전체 법률 텍스트
            'chapters': []  # 장/절 목록을 저장할 리스트
        }
        
        # 제목 패턴 (# 또는 ## 로 시작하는 라인) - 마크다운 형식의 제목 인식
        title_pattern = r'^(#+)\s+(.+)$'
        
        # 조문 패턴 (제X조) - 법률 조문 형식 인식 (예: 제1조(목적) 이 법은...)
        article_pattern = r'제(\d+)조\(([^)]+)\)\s+(.+)'
        
        # 항 패턴 (①, ②, ③ 등으로 시작) - 법률 항 형식 인식
        paragraph_pattern = r'[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮]\s+(.+)'
        
        # 호 패턴 (1., 2., 3. 등으로 시작) - 법률 호 형식 인식
        subparagraph_pattern = r'[-\s]*(\d+)\.\s+[\'"]?(.+?)[\'"]?$'
        
        lines = content.split('\n')  # 줄 단위로 분리
        current_chapter = None  # 현재 처리 중인 장/절
        current_article = None  # 현재 처리 중인 조문
        current_paragraph = None  # 현재 처리 중인 항
        
        for line in lines:
            # 장/절 매칭 - 마크다운 제목 형식 확인
            chapter_match = re.match(title_pattern, line)
            if chapter_match:
                level = len(chapter_match.group(1))  # # 개수로 제목 레벨 결정
                title = chapter_match.group(2).strip()  # 제목 텍스트 추출
                
                if '장' in title or '절' in title:  # 장 또는 절이 포함된 제목인 경우
                    current_chapter = {
                        'title': title,  # 장/절 제목
                        'type': 'chapter',  # 노드 타입
                        'full_text': title,  # 전체 텍스트 (초기값은 제목)
                        'articles': []  # 조문 목록을 저장할 리스트
                    }
                    law_structure['chapters'].append(current_chapter)  # 법률 구조에 장/절 추가
                    current_article = None  # 새 장/절로 이동했으므로 현재 조문 초기화
                    current_paragraph = None  # 현재 항 초기화
                continue
            
            # 조문 매칭 - 법률 조문 형식 확인
            article_match = re.search(article_pattern, line)
            if article_match:
                article_num = article_match.group(1)  # 조문 번호 추출
                article_title = article_match.group(2)  # 조문 제목 추출
                article_content = article_match.group(3)  # 조문 내용 추출
                
                article_full_title = f'제{article_num}조({article_title})'  # 전체 조문 제목 생성
                
                current_article = {
                    'number': article_num,  # 조문 번호
                    'title': article_full_title,  # 조문 전체 제목
                    'content': article_content,  # 조문 내용
                    'full_text': f"{article_full_title} {article_content}",  # 조문 전체 텍스트
                    'type': 'article',  # 노드 타입
                    'paragraphs': []  # 항 목록을 저장할 리스트
                }
                
                # 조문이 속한 장이 없으면 기본 장 생성 (장이 명시되지 않은 조문을 위함)
                if current_chapter is None:
                    current_chapter = {
                        'title': '기본',  # 기본 장 제목
                        'type': 'chapter',  # 노드 타입
                        'full_text': '기본',  # 전체 텍스트
                        'articles': []  # 조문 목록
                    }
                    law_structure['chapters'].append(current_chapter)  # 법률 구조에 기본 장 추가
                
                current_chapter['articles'].append(current_article)  # 현재 장에 조문 추가
                current_paragraph = None  # 새 조문으로 이동했으므로 현재 항 초기화
                
                # 장의 full_text 업데이트 - 조문 정보 추가
                current_chapter['full_text'] += f"\n{current_article['full_text']}"
                continue
            
            # 항 매칭 - 법률 항 형식 확인
            paragraph_match = re.match(paragraph_pattern, line)
            if paragraph_match and current_article:  # 현재 조문이 있는 경우에만 항 처리
                paragraph_content = paragraph_match.group(1)  # 항 내용 추출
                
                current_paragraph = {
                    'content': paragraph_content,  # 항 내용
                    'full_text': paragraph_content,  # 항 전체 텍스트
                    'type': 'paragraph',  # 노드 타입
                    'subparagraphs': []  # 호 목록을 저장할 리스트
                }
                
                current_article['paragraphs'].append(current_paragraph)  # 현재 조문에 항 추가
                
                # 조문의 full_text 업데이트 - 항 정보 추가
                current_article['full_text'] += f"\n{paragraph_content}"
                # 장의 full_text 업데이트 - 항 정보 추가
                current_chapter['full_text'] += f"\n{paragraph_content}"
                continue
            
            # 호 매칭 - 법률 호 형식 확인
            subparagraph_match = re.match(subparagraph_pattern, line)
            if subparagraph_match and current_paragraph:  # 현재 항이 있는 경우에만 호 처리
                subparagraph_num = subparagraph_match.group(1)  # 호 번호 추출
                subparagraph_content = subparagraph_match.group(2)  # 호 내용 추출
                
                subparagraph = {
                    'number': subparagraph_num,  # 호 번호
                    'content': subparagraph_content,  # 호 내용
                    'full_text': f"{subparagraph_num}. {subparagraph_content}",  # 호 전체 텍스트
                    'type': 'subparagraph'  # 노드 타입
                }
                
                current_paragraph['subparagraphs'].append(subparagraph)  # 현재 항에 호 추가
                
                # 항의 full_text 업데이트 - 호 정보 추가
                current_paragraph['full_text'] += f"\n{subparagraph['full_text']}"
                # 조문의 full_text 업데이트 - 호 정보 추가
                current_article['full_text'] += f"\n{subparagraph['full_text']}"
                # 장의 full_text 업데이트 - 호 정보 추가
                current_chapter['full_text'] += f"\n{subparagraph['full_text']}"
        
        return law_structure

    def create_knowledge_graph(self):
        """
        지식 그래프 생성 - 법, 장, 조, 항, 호 간의 계층적 관계 구축
        
        Returns:
            bool: 그래프 생성 성공 여부
        """
        # 법률 구조 추출
        all_laws = self.extract_structure()
        
        # 노드 및 관계 생성
        for law_name, law_structure in tqdm(all_laws.items(), desc="법률 온톨로지 구축 중"):
            # 법률 노드 생성
            law_id = f"law_{law_name}"  # 법률 노드 고유 ID 생성
            law_node = Node(
                id=law_id,  # 노드 ID
                type="Law",  # 노드 타입 (법률)
                properties={
                    "name": law_name,  # 법률 이름
                    "full_text": law_structure['full_text']  # 법률 전체 텍스트
                }
            )
            self.node_dict[law_id] = law_node  # 노드 사전에 법률 노드 추가
            
            # 장 노드 생성 및 법률과 연결
            for chapter in law_structure['chapters']:
                chapter_id = f"chapter_{law_name}_{chapter['title']}"  # 장 노드 고유 ID 생성
                chapter_node = Node(
                    id=chapter_id,  # 노드 ID
                    type="Chapter",  # 노드 타입 (장)
                    properties={
                        "title": chapter['title'],  # 장 제목
                        "full_text": chapter['full_text']  # 장 전체 텍스트
                    }
                )
                self.node_dict[chapter_id] = chapter_node  # 노드 사전에 장 노드 추가
                
                # 법률과 장 연결 - CONTAINS 관계 생성
                self.relationships.append(
                    Relationship(
                        source=self.node_dict[law_id],  # 출발 노드 (법률)
                        target=self.node_dict[chapter_id],  # 도착 노드 (장)
                        type="CONTAINS",  # 관계 타입 (포함)
                        properties={}  # 관계 속성 (없음)
                    )
                )
                
                # 조문 노드 생성 및 장과 연결
                prev_article_node = None  # 이전 조문 노드 추적용 변수
                
                for article in chapter['articles']:
                    article_id = f"article_{law_name}_{article['title']}"  # 조문 노드 고유 ID 생성
                    article_node = Node(
                        id=article_id,  # 노드 ID
                        type="Article",  # 노드 타입 (조문)
                        properties={
                            "title": article['title'],  # 조문 제목
                            "content": article['content'],  # 조문 내용
                            "number": article['number'],  # 조문 번호
                            "full_text": article['full_text']  # 조문 전체 텍스트
                        }
                    )
                    self.node_dict[article_id] = article_node  # 노드 사전에 조문 노드 추가
                    
                    # 장과 조문 연결 - CONTAINS 관계 생성
                    self.relationships.append(
                        Relationship(
                            source=self.node_dict[chapter_id],  # 출발 노드 (장)
                            target=self.node_dict[article_id],  # 도착 노드 (조문)
                            type="CONTAINS",  # 관계 타입 (포함)
                            properties={}  # 관계 속성 (없음)
                        )
                    )
                    
                    # 이전 조문과 현재 조문 간의 NEXT_TO 관계 생성
                    if prev_article_node:
                        self.relationships.append(
                            Relationship(
                                source=prev_article_node,  # 출발 노드 (이전 조문)
                                target=self.node_dict[article_id],  # 도착 노드 (현재 조문)
                                type="NEXT_TO",  # 관계 타입 (다음)
                                properties={}  # 관계 속성 (없음)
                            )
                        )
                    
                    # 현재 조문을 이전 조문으로 설정
                    prev_article_node = self.node_dict[article_id]
                    
                    # 항 노드 생성 및 조문과 연결
                    prev_paragraph_node = None  # 이전 항 노드 추적용 변수
                    
                    for i, paragraph in enumerate(article['paragraphs']):
                        paragraph_id = f"paragraph_{law_name}_{article['title']}_{i+1}"  # 항 노드 고유 ID 생성
                        paragraph_node = Node(
                            id=paragraph_id,  # 노드 ID
                            type="Paragraph",  # 노드 타입 (항)
                            properties={
                                "content": paragraph['content'],  # 항 내용
                                "number": i+1,  # 항 번호 (인덱스 기반)
                                "full_text": paragraph['full_text']  # 항 전체 텍스트
                            }
                        )
                        self.node_dict[paragraph_id] = paragraph_node  # 노드 사전에 항 노드 추가
                        
                        # 조문과 항 연결 - CONTAINS 관계 생성
                        self.relationships.append(
                            Relationship(
                                source=self.node_dict[article_id],  # 출발 노드 (조문)
                                target=self.node_dict[paragraph_id],  # 도착 노드 (항)
                                type="CONTAINS",  # 관계 타입 (포함)
                                properties={}  # 관계 속성 (없음)
                            )
                        )
                        
                        # 이전 항과 현재 항 간의 NEXT_TO 관계 생성
                        if prev_paragraph_node:
                            self.relationships.append(
                                Relationship(
                                    source=prev_paragraph_node,  # 출발 노드 (이전 항)
                                    target=self.node_dict[paragraph_id],  # 도착 노드 (현재 항)
                                    type="NEXT_TO",  # 관계 타입 (다음)
                                    properties={}  # 관계 속성 (없음)
                                )
                            )
                        
                        # 현재 항을 이전 항으로 설정
                        prev_paragraph_node = self.node_dict[paragraph_id]
                        
                        # 호 노드 생성 및 항과 연결
                        prev_subparagraph_node = None  # 이전 호 노드 추적용 변수
                        
                        for subparagraph in paragraph['subparagraphs']:
                            subparagraph_id = f"subparagraph_{law_name}_{article['title']}_{i+1}_{subparagraph['number']}"  # 호 노드 고유 ID 생성
                            subparagraph_node = Node(
                                id=subparagraph_id,  # 노드 ID
                                type="Subparagraph",  # 노드 타입 (호)
                                properties={
                                    "content": subparagraph['content'],  # 호 내용
                                    "number": subparagraph['number'],  # 호 번호
                                    "full_text": subparagraph['full_text']  # 호 전체 텍스트
                                }
                            )
                            self.node_dict[subparagraph_id] = subparagraph_node  # 노드 사전에 호 노드 추가
                            
                            # 항과 호 연결 - CONTAINS 관계 생성
                            self.relationships.append(
                                Relationship(
                                    source=self.node_dict[paragraph_id],  # 출발 노드 (항)
                                    target=self.node_dict[subparagraph_id],  # 도착 노드 (호)
                                    type="CONTAINS",  # 관계 타입 (포함)
                                    properties={}  # 관계 속성 (없음)
                                )
                            )
                            
                            # 이전 호와 현재 호 간의 NEXT_TO 관계 생성
                            if prev_subparagraph_node:
                                self.relationships.append(
                                    Relationship(
                                        source=prev_subparagraph_node,  # 출발 노드 (이전 호)
                                        target=self.node_dict[subparagraph_id],  # 도착 노드 (현재 호)
                                        type="NEXT_TO",  # 관계 타입 (다음)
                                        properties={}  # 관계 속성 (없음)
                                    )
                                )
                            
                            # 현재 호를 이전 호로 설정
                            prev_subparagraph_node = self.node_dict[subparagraph_id]
        
        # GraphDocument 객체 생성 - Neo4j에 저장하기 위한 형식
        nodes = list(self.node_dict.values())  # 모든 노드 목록
        graph_doc = GraphDocument(
            nodes=nodes,  # 노드 목록
            relationships=self.relationships  # 관계 목록
        )
        
        # 기존 데이터 삭제 - 데이터베이스 초기화
        # self.graph.query("MATCH (n) DETACH DELETE n")
        
        # 생성된 GraphDocument를 Neo4j 데이터베이스에 저장
        self.graph.add_graph_documents([graph_doc])
        
        print(f"총 노드 수: {len(self.node_dict)}")
        print(f"총 관계 수: {len(self.relationships)}")
        print("법률 온톨로지 구축 완료!")
        
        return True



#마크다운 파일에서 법률 문서 구조 추출하는 함수
def extract_law_structure(law_contents):
    """
    법률 문서 구조를 추출하는 함수
    
    Args :
        law_contents (dict) : 법률 문서 내용을 담은 딕셔너리
    
    Returns :
        dict : 법률별로 계층적 구조를 가진 딕셔너리
    """
    extractor = LaborLawKGExtractor(law_contents)
    all_laws = extractor.extract_structure()
    return all_laws    

#법률 문서 구조 추출
all_laws = extract_law_structure(law_contents)    

#추출된 문서 구조 확인
print(f"추출된 법률 문서 수: {len(all_laws)}")
first_law_name = list(all_laws.keys())[0] # 첫 번째 법률 이름 가져오기
print(f"첫 번째 법률: {first_law_name}")
print(f"첫 번째 법률의 장 수: {len(all_laws[first_law_name]['chapters'])}")

추출된 법률 문서 수: 3
첫 번째 법률: 근로기준법 시행규칙(고용노동부령)(제00436호)(20250223)
첫 번째 법률의 장 수: 1


In [7]:
#지식 그래프 구축
extractor = LaborLawKGExtractor(law_contents)
extractor.create_knowledge_graph()

법률 온톨로지 구축 중: 100%|██████████| 3/3 [00:00<00:00, 479.51it/s]


총 노드 수: 237
총 관계 수: 429
법률 온톨로지 구축 완료!


True

In [8]:
#지식 그래프 확인
graph_db = extractor.graph

#모든 노드 조회
graph_db.query("MATCH (n) RETURN count(n)")

[{'count(n)': 237}]

### 3.3 법령 간의 관계를 추가

- **법률-시행령**, **시행령-시행규칙** 간의 관계를 정의
- 각 관계는 **HAS_DECREE**, **HAS_RULE** 유형으로 Neo4j 쿼리문을 통해 생성

In [9]:
#법률과 시행령 간의 관계 추가
law_decree_query = """
MATCH (law:Law), (decree: Law)
WHERE law.name CONTAINS '근로기준법(법률)'
AND decree.name CONTAINS '근로기준법 시행령(대통령령)'
CREATE (law)-[r:HAS_DECREE]->(decree)
RETURN count(r) as relationships_created
"""

#시행령과 시행규칙 간의 관계 추가
decree_rule_query ="""
MATCH (decree:Law), (rule:Law)
WHERE decree.name CONTAINS '근로기준법 시행령(대통령령)'
AND rule.name CONTAINS '근로기준법 시행규칙(고용노동부령)'
CREATE (decree)-[r:HAS_RULE]->(rule)
RETURN count(r) as relationships_created
"""

#쿼리 실행 및 결과 확인
law_decree_result = graph_db.query(law_decree_query)
decree_rule_result = graph_db.query(decree_rule_query)

print(f"법률-시행령 관계 생성: {law_decree_result[0]['relationships_created']}개")
print(f"시행령-시행규칙 관계 생성: {decree_rule_result[0]['relationships_created']}개")

법률-시행령 관계 생성: 1개
시행령-시행규칙 관계 생성: 1개


---

## 4. **Graph RAG 구현**

### 4.1 벡터 저장소에 인덱싱

- 법률 조문을 위한 **벡터 인덱스**를 추가
- **벡터 임베딩** 생성 및 Neo4j 저장 기능 구현

In [42]:
import google.generativeai as genai
import os

# API 키 설정
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

print("--- 사용 가능한 임베딩 모델 리스트 ---")
for m in genai.list_models():
    # 'embedContent'가 가능한 모델만 필터링
    if 'embedContent' in m.supported_generation_methods:
        print(f"모델명(name): {m.name}")

C:\Users\yoonj\AppData\Local\Temp\ipykernel_12104\2022803790.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


--- 사용 가능한 임베딩 모델 리스트 ---
모델명(name): models/gemini-embedding-001


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Google 임베딩 모델 초기화
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", 
                                          api_key=os.getenv("GOOGLE_API_KEY"),
                                          output_dimensionality=768,
                                          task_type="retrieval_document")

try:
    test_emb = embeddings.embed_query("테스트")
    print(f"✅ 연결 성공! 생성된 벡터 차원: {len(test_emb)}")
except Exception as e:
    print(f"❌ 실패: {e}")

✅ 연결 성공! 생성된 벡터 차원: 768


#### 2) **벡터 인덱스** 생성

- 각 조문 노드의 **content_embedding** 필드에 적용하고 벡터 인덱스를 개별 생성함 (각 레이블 별로 별도 인덱스 생성만 가능)
- 벡터 차원을 **1536차원**으로 설정하여 OpenAI의 text-embedding-3-small 모델과 호환되도록 함

In [11]:
# 법률/시행령/시행규칙 조항 벡터 인덱스 생성
create_law_index_query = """
CREATE VECTOR INDEX law_article_embeddings IF NOT EXISTS
FOR (n:Article)
ON n.content_embedding
OPTIONS {indexConfig: {
  `vector.dimensions`: 768,
  `vector.similarity_function`: 'cosine'
}}
"""

# 모든 벡터 인덱스 생성 쿼리 실행
graph.query(create_law_index_query)

[]

In [72]:
#벡터 인덱스 확인
check_vector_index_query = """
SHOW VECTOR INDEXES
"""

vector_indexes = graph.query(check_vector_index_query)
for index in vector_indexes:
    #벡터 인덱스 정보 출력
    print(f"Index Name: {index['name']}")
    print(f"Type: {index['type']}")
    print(f"Property Key : {index['properties']}")
    print("-"*40)

Index Name: law_article_embeddings
Type: VECTOR
Property Key : ['content_embedding']
----------------------------------------


#### 3) **임베딩 생성 및 저장**

- 텍스트에 대해 **OpenAI 임베딩**을 생성하는 과정 수행
- 빈 문자열인 경우 처리를 **건너뛰는** 예외 처리 포함
- 생성된 임베딩을 `db.create.setNodeVectorProperty` 프로시저를 통해 **content_embedding** 속성으로 저장

In [12]:
import time
# 법률 조항 데이터 가져오기
law_query = """
MATCH (a:Article)
WHERE a.content IS NOT NULL
RETURN a.id AS id, a.title AS title, a.content AS content
"""
law_articles = graph.query(law_query)

# 배치 크기 설정 (안정성을 위해 50~100 사이 권장)
BATCH_SIZE = 20 

for i in range(0, len(law_articles), BATCH_SIZE):
    batch = law_articles[i:i+BATCH_SIZE]
    batch_texts = []
    batch_ids = []
    
    for article in batch:
        content_text = f"{article['title']}\n\n{article['content']}"
        if content_text.strip():
            batch_texts.append(content_text)
            batch_ids.append(article['id'])
    
    # 최대 3번까지 재시도하는 로직
    for attempt in range(3):
        try:
            if batch_texts:
                batch_embeddings = embeddings.embed_documents(batch_texts)
                
                batch_data = [{"id": aid, "embedding": emb} 
                              for aid, emb in zip(batch_ids, batch_embeddings)]
                
                batch_update_query = """
                UNWIND $batch AS item
                MATCH (a:Article {id: item.id})
                CALL db.create.setNodeVectorProperty(a, 'content_embedding', item.embedding)
                RETURN count(a) as updated
                """
                
                result = graph.query(batch_update_query, params={"batch": batch_data})
                print(f"✅ 배치 완료: {i+1}~{min(i+len(batch_texts), len(law_articles))} / 업데이트: {result[0]['updated']}")
                
                # Free Tier 안정성을 위해 배치 간 5~10초 정도 대기
                time.sleep(10) 
                break # 성공 시 시도 루프 탈출
                
        except Exception as e:
            if "429" in str(e):
                wait_time = 40  # 429 에러 발생 시 40초 대기
                print(f"⚠️ 할당량 초과! {wait_time}초 대기 후 재시도합니다... (시도 {attempt+1}/3)")
                time.sleep(wait_time)
            else:
                print(f"❌ 기타 에러 발생: {e}")
                break

print(f"법률 조항 임베딩 업데이트 최종 완료!!")

✅ 배치 완료: 1~20 / 업데이트: 20
✅ 배치 완료: 21~40 / 업데이트: 20
✅ 배치 완료: 41~60 / 업데이트: 20
✅ 배치 완료: 61~80 / 업데이트: 20
✅ 배치 완료: 81~100 / 업데이트: 20
✅ 배치 완료: 101~120 / 업데이트: 20
✅ 배치 완료: 121~140 / 업데이트: 20
✅ 배치 완료: 141~160 / 업데이트: 20
✅ 배치 완료: 161~180 / 업데이트: 20
✅ 배치 완료: 181~196 / 업데이트: 16
법률 조항 임베딩 업데이트 최종 완료!!


### 4.2 RAG 시스템 구현


#### 1) **Neo4j Graph DB 검색 설정**

In [13]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_neo4j import Neo4jVector

# Google 임베딩 모델 초기화
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",
                                          output_dimensionality=768,task_type="retrieval_query")

#Neo4j 데이터베이스에 이미 생성된 벡터 인덱스에 연결하는 Neo4jVector 인스턴스 생성
vector_store = Neo4jVector.from_existing_index(
    embeddings, #사용할 임베딩 모델 지정
    url = os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD"),  
    database=os.getenv("NEO4J_DATABASE"), 
    index_name = "law_article_embeddings", #법률 데이터용 벡터 인덱스 이름
    embedding_dimension=768,
    node_label = "Article",
    text_node_property="content",
    embedding_node_property = "content_embedding"
)

In [14]:
#벡터 검색 테스트
query = "연차휴가는 연간 몇일을 부여해야 하나요?"
results = vector_store.similarity_search_with_score(query=query, k=5)

print(f"검색어: '{query}'에 대한 결과")
print("-"*50)

for i, (doc,score) in enumerate(results):
    similarity = 1-score #코사인 거리를 유사도로 변환
    print(f"\n결과 #{i+1} (유사도: {similarity:.4f})")
    print(f"제목: {doc.metadata.get('title','제목없음')}")
    print(f"내용 미리보기: {doc.page_content[:150]}...")

검색어: '연차휴가는 연간 몇일을 부여해야 하나요?'에 대한 결과
--------------------------------------------------

결과 #1 (유사도: 0.1106)
제목: 제60조(연차 유급휴가)
내용 미리보기: ① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다. &lt;개정 2012. 2. 1.&gt;...

결과 #2 (유사도: 0.1409)
제목: 제62조(유급휴가의 대체)
내용 미리보기: 사용자는 근로자대표와의 서면 합의에 따라 제60조에 따른 연차 유급휴가일을 갈음하여 특정 한 근로일에 근로자를 휴무시킬 수 있다....

결과 #3 (유사도: 0.1441)
제목: 제55조(휴일)
내용 미리보기: ① 사용자는 근로자에게 1주에 평균 1회 이상의 유급휴일을 보장하여야 한다. &lt;개정 2018. 3. 20.&gt;...

결과 #4 (유사도: 0.1473)
제목: 제61조(연차 유급휴가의 사용 촉진)
내용 미리보기: ① 사용자가 제60조제1항ㆍ제2항 및 제4항에 따른 유급휴가(계속하여 근로한 기간 이 1년 미만인 근로자의 제60조제2항에 따른 유급휴가는 제외한다)의 사용을 촉진하기 위하여 다음 각 호의 조치를 하였음에도 불구하고 근로자가 휴가를 사용하지 아니하여 제60조제7항 본문...

결과 #5 (유사도: 0.1491)
제목: 제30조(휴일)
내용 미리보기: ① 법 제55조제1항에 따른 유급휴일은 1주 동안의 소정근로일을 개근한 자에게 주어야 한다. &lt;개정 2018. 6. 29.&gt;...


In [15]:
print(f"검색 결과 개수: {len(results)}") # 이게 0인지 확인

for i, (doc, score) in enumerate(results):
    print(f"결과 찾음: {doc.metadata.get('title')}")

검색 결과 개수: 5
결과 찾음: 제60조(연차 유급휴가)
결과 찾음: 제62조(유급휴가의 대체)
결과 찾음: 제55조(휴일)
결과 찾음: 제61조(연차 유급휴가의 사용 촉진)
결과 찾음: 제30조(휴일)


In [16]:
doc.metadata

{'title': '제30조(휴일)',
 'number': '30',
 'full_text': '제30조(휴일) ① 법 제55조제1항에 따른 유급휴일은 1주 동안의 소정근로일을 개근한 자에게 주어야 한다. &lt;개정 2018. 6. 29.&gt;'}

#### 2) **벡터 검색 기반 RAG 구현**

In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough

#법률 데이터를 위한 RAG 프롬프트 템플릿 정의
template = """
당신은 한국 법률 전문가 AI 비서입니다.
제공된 법률 조항 내용을 바탕으로 질문에 정확하게 답변해 주세요. 출처를 반드시 표기해 주세요. (예: 출처: 법률 조항 제목)
법률 조항에서 찾을 수 없는 정보에 대해서는 솔직하게 모른다고 답변하세요.
법적 조언이 필요한 경우에는 전문 법률 상담을 권유하세요.

참고할 법률 조항:
{context}

질문: {question}

답변 : 

"""

prompts = PromptTemplate(
    template = template,
    input_variables=["context","question"]
)

#법률 데이터 RAG체인 구성
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k":5})

law_rag_chain ={
    "question" : RunnablePassthrough(),
    "context" : retriever
} | prompts | llm | StrOutputParser()


#법률 RAG 실행 예시
def law_assistant(query):
    """법률 질문에 대한 답변을 제공하는 함수"""
    response = law_rag_chain.invoke(query)
    print(f"질문: {query}\n\n답변: {response}\n{'-'*70}")
    return response

#테스트 질문 예시 - 근로기준법 관련
test_question =[
    "법정 근로시간은 어떻게 되나요?",
    "주휴수당의 계산 방법과 지급 기준은 무엇인가요?",
    "해고 예고 제도의 내용과 예외는 무엇인가요?",
    "연차유급휴가 발생 요건과 일수 계산 방법은 어떻게 되나요?"
]
    # "직장 내 괴롭힘 금지에 관한 규정은 무엇인가요?",
    # "임금 체불 시 사업주에 대한 제재 규정은 어떻게 되나요?",
    # "출산 전후 휴가와 육아휴직 제도의 주요 내용은 무엇인가요?",
    # "근로계약서 작성 의무와 필수 기재사항은 무엇인가요?"
#테스트 질문 실행
for question in test_question:
    law_assistant(question)

질문: 법정 근로시간은 어떻게 되나요?

답변: 제공된 법률 조항에 따르면 법정 근로시간은 다음과 같습니다.

1.  **일반적인 근로시간:**
    *   1주 간의 근로시간은 휴게시간을 제외하고 40시간을 초과할 수 없습니다.
    *   출처: 제50조(근로시간)

2.  **15세 이상 18세 미만인 사람의 근로시간:**
    *   1일에 7시간, 1주에 35시간을 초과하지 못합니다.
    *   다만, 당사자 사이의 합의에 따라 1일에 1시간, 1주에 5시간을 한도로 연장할 수 있습니다.
    *   출처: 제69조(근로시간)

3.  **근로시간 계산 시 휴게시간 제외:**
    *   위 근로시간은 휴게시간을 제외한 근로시간을 말합니다.
    *   출처: 제41조(근로시간의 계산), 제50조(근로시간)

4.  **탄력적 근로시간제 (3개월 이내):**
    *   사용자는 취업규칙에 따라 2주 이내의 일정한 단위기간을 평균하여 1주 간의 근로시간이 40시간을 초과하지 아니하는 범위에서 특정한 주에 40시간을, 특정한 날에 제50조제2항의 근로시간을 초과하여 근로하게 할 수 있습니다.
    *   다만, 특정한 주의 근로시간은 48시간을 초과할 수 없습니다.
    *   출처: 제51조(3개월 이내의 탄력적 근로시간제)
    *   *참고: 제공된 법률 조항에는 제50조제2항의 내용이 포함되어 있지 않아, 탄력적 근로시간제 적용 시 특정한 날의 근로시간 상한은 확인할 수 없습니다.*

5.  **근로시간 및 휴게시간의 특례:**
    *   특정 산업에 해당하는 사업의 경우, 사용자가 근로자대표와 서면으로 합의하면 주 12시간을 초과하는 연장근로를 하게 하거나 휴게시간을 변경할 수 있습니다. 이는 법정 근로시간 자체의 변경이라기보다는 연장근로 한도에 대한 특례 조항입니다.
    *   출처: 제59조(근로시간 및 휴게시간의 특례)

법적 조언이 필요한 경우에는 전문 법률 상담을 권유합니다.
-----------------

#### 3) **지식 그래프 강화 RAG 구현**

In [19]:
def kg_enhanced_law_rag(question):
    """지식 그래프 정보로 강화된 법률 RAG 시스템"""
    try:
        # 1. 벡터 검색으로 관련 법률 문서 찾기
        docs = vector_store.similarity_search(question, k=5)

        #검색된 문서의 ID 추출
        doc_ids =[]
        for doc in docs:
            if "id" in doc.metadata:
                doc_ids.append(doc.metadata["id"])
            elif "title" in doc.metadata:
                doc_ids.append(doc.metadata["title"])
        
        #검색 결과가 없는 경우 처리
        if not doc_ids:
            return {
                "query": question,
                "result": "관련 법률 조항을 찾을 수 없습니다. 다른 질문을 시도해 보세요.",
                "intermediate_steps":[]
            }
        
        # 2. 그래프 검색 : 가변 경로를 사용하여 관련 조문과 그 전후 조문 찾기 (2단계 깊이까지)
        cypher_query = """
        //벡터 검색 결과와 일치하는 법률 조문 찾기
        MATCH (article:Article)
        WHERE article.id in $doc_ids OR article.title IN $doc_ids

        //가변 경로를 사용하여 1~2단계 깊이의 조문 찾기
        MATCH (article)-[r*1..2]->(related:Article)

        //결과 수집 밀 가공
        WITH article, related,r
        WITH article,
             related,
             size(r) AS path_length,
             [rel IN r | type(rel)] AS relationship_types
        
        // 최종 결과 반환
        RETURN article.id AS article_id,
               article.title AS title,
               article.content AS content,
               COLLECT(DISTINCT {
                   id: related.id, 
                   title: related.title, 
                   content: related.content, 
                   path_length: path_length,
                   relationships: relationship_types
               
               }) AS relatedArticles
        """

        #그래프 검색 실행 및 결과 처리
        graph_results = graph.query(cypher_query, {"doc_ids": doc_ids})

        # 3. 그래프 정보를 텍스트로 변환
        kg_context =""
        for record in graph_results:
            kg_context += f"조문: {record['title']}\n내용: {record['content']}\n\n"

            #관련 조문 정보 추가 (경로 길이에 따라 구분)
            kg_context += "관련 조문:\n"

            #1단계와 2단계 관계를 구분하여 표시
            level1_articles = [rel for rel in record['relatedArticles'] if rel['path_length'] == 1]
            level2_articles = [rel for rel in record['relatedArticles'] if rel['path_length'] == 2]

            #1단계 관련 조문 표시
            kg_context += "직접 관련 조문: \n"
            for article in level1_articles:
                rel_type = article['relationships'][0] if article['relationships'] else "관계 없음"
                kg_context += f"- {article['title']} (관계: {rel_type})\n {article['content']}\n\n"
            
            # 2단계 관련 조문 표시
            kg_context += "간접 관련 조문:\n"
            for article in level2_articles:
                rel_types = " -> ".join(article['relationships']) if article['relationships'] else "관계 없음"
                kg_context += f"- {article['title']} (관계 경로: {rel_types})\n  {article['content']}\n\n"
        
        # 4. 원본 문서 내용 가져오기
        doc_context = "\n".join([f"{doc.metadata.get('title','제목 없음')}: {doc.page_content}" for doc in docs])

        # 5. 통합 컨텍스트 생성
        combined_context = f"법률 문서 정보:\n{doc_context}\n\n법률 지식 그래프 정보:\n{kg_context}"

        # 6. 프롬프트 템플릿 정의
        kg_template="""        
        당신은 근로기준법에 대한 전문 지식을 갖춘 법률 전문가입니다.
        제공된 법률 조문과 지식 그래프 정보를 바탕으로 질문에 정확하게 답변해 주세요.
        
        지식 그래프는 법률 조문 간의 관계와 연결성을 보여줍니다.
        이 관계 정보를 활용하여 더 정확하고 포괄적인 법률 해석을 제공하세요.
        
        법률 조문이나 지식 그래프에서 찾을 수 없는 정보에 대해서는 솔직하게 모른다고 답변하세요.
        답변은 법률 용어를 적절히 사용하되, 일반인도 이해할 수 있도록 명확하게 작성해 주세요.
        
        참고할 정보 :
        {context}

        질문 : {question}

        답변 :
        """

        kg_prompt = PromptTemplate(
            template=kg_template,
            input_variables=["context","question"]
        )

        # 7. RAG 체인 구성 및 실행
        rag_chain = kg_prompt | llm | StrOutputParser()
        result = rag_chain.invoke({
            "question": question,
            "context": combined_context
        })

        #8. 중간 단계 정보 포함하여 결과 반환
        intermediate_steps =[
            {"query": cypher_query},
            {"context": [dict(record) for record in graph_results]}
        ]

        return {
            "query": question,
            "result": result,
            "intermediate_steps": intermediate_steps
        }
    
    except Exception as e:
        #오류 처리 및 디버깅 정보 반환
        return {
            "query": question,
            "result": f"검색 중 오류가 발생했습니다. {str(e)}",
            "error" : str(e)
        }
    

In [20]:
#실행 테스트
result = kg_enhanced_law_rag("법정 근로시간은 어떻게 되나요?")
print(result["result"])

근로기준법상 법정 근로시간은 근로자의 연령 및 적용되는 제도에 따라 다음과 같이 구분됩니다.

1.  **일반적인 근로자의 법정 근로시간:**
    *   근로기준법 **제50조제1항**에 따라, 1주 간의 근로시간은 휴게시간을 제외하고 40시간을 초과할 수 없습니다.

2.  **연소 근로자의 법정 근로시간:**
    *   근로기준법 **제69조**에 따르면, 15세 이상 18세 미만인 사람의 근로시간은 1일에 7시간, 1주에 35시간을 초과하지 못합니다.
    *   다만, 당사자 간의 합의가 있는 경우에는 1일에 1시간, 1주에 5시간을 한도로 연장근로를 할 수 있습니다.

3.  **근로시간 계산 원칙:**
    *   근로기준법 **제41조** 및 **제50조**에 명시된 바와 같이, 근로시간을 계산할 때에는 휴게시간은 제외됩니다. 이는 연소 근로자의 근로시간(제69조) 계산에도 동일하게 적용됩니다.

4.  **예외적인 근로시간 제도:**
    *   **탄력적 근로시간제 (제51조):** 근로기준법 **제50조**와 직접적으로 관련된 **제51조**에 따라, 사용자는 취업규칙 등에서 정하는 바에 따라 2주 이내의 일정한 단위기간을 평균하여 1주 간의 근로시간이 **제50조제1항**의 근로시간(40시간)을 초과하지 않는 범위 내에서, 특정한 주에 40시간을, 특정한 날에 8시간을 초과하여 근로하게 할 수 있습니다. 다만, 특정한 주의 근로시간은 48시간을 초과할 수 없습니다.
    *   **근로시간 및 휴게시간의 특례 (제59조):** 「통계법」에 따른 특정 산업에 해당하는 사업의 경우, 사용자가 근로자대표와 서면으로 합의하면 주 12시간을 초과하여 연장근로를 하게 하거나 휴게시간을 변경할 수 있는 예외가 적용될 수 있습니다.

이처럼 법정 근로시간은 기본 원칙과 함께 근로자의 특성(연소 근로자)이나 사업장의 특성(탄력적 근로시간제, 특례 사업)에 따라 다르게 적용될 수 있습니다.


In [21]:
from pprint import pprint
pprint(result['result'])

('근로기준법상 법정 근로시간은 근로자의 연령 및 적용되는 제도에 따라 다음과 같이 구분됩니다.\n'
 '\n'
 '1.  **일반적인 근로자의 법정 근로시간:**\n'
 '    *   근로기준법 **제50조제1항**에 따라, 1주 간의 근로시간은 휴게시간을 제외하고 40시간을 초과할 수 없습니다.\n'
 '\n'
 '2.  **연소 근로자의 법정 근로시간:**\n'
 '    *   근로기준법 **제69조**에 따르면, 15세 이상 18세 미만인 사람의 근로시간은 1일에 7시간, 1주에 35시간을 '
 '초과하지 못합니다.\n'
 '    *   다만, 당사자 간의 합의가 있는 경우에는 1일에 1시간, 1주에 5시간을 한도로 연장근로를 할 수 있습니다.\n'
 '\n'
 '3.  **근로시간 계산 원칙:**\n'
 '    *   근로기준법 **제41조** 및 **제50조**에 명시된 바와 같이, 근로시간을 계산할 때에는 휴게시간은 제외됩니다. 이는 '
 '연소 근로자의 근로시간(제69조) 계산에도 동일하게 적용됩니다.\n'
 '\n'
 '4.  **예외적인 근로시간 제도:**\n'
 '    *   **탄력적 근로시간제 (제51조):** 근로기준법 **제50조**와 직접적으로 관련된 **제51조**에 따라, 사용자는 '
 '취업규칙 등에서 정하는 바에 따라 2주 이내의 일정한 단위기간을 평균하여 1주 간의 근로시간이 **제50조제1항**의 '
 '근로시간(40시간)을 초과하지 않는 범위 내에서, 특정한 주에 40시간을, 특정한 날에 8시간을 초과하여 근로하게 할 수 있습니다. '
 '다만, 특정한 주의 근로시간은 48시간을 초과할 수 없습니다.\n'
 '    *   **근로시간 및 휴게시간의 특례 (제59조):** 「통계법」에 따른 특정 산업에 해당하는 사업의 경우, 사용자가 '
 '근로자대표와 서면으로 합의하면 주 12시간을 초과하여 연장근로를 하게 하거나 휴게시간을 변경할 수 있는 예외가 적용될 수 있습니다.\n'
 '\n'
 '이처럼 법정 근로시간

In [22]:
pprint(result['intermediate_steps'])

[{'query': '\n'
           '        //벡터 검색 결과와 일치하는 법률 조문 찾기\n'
           '        MATCH (article:Article)\n'
           '        WHERE article.id in $doc_ids OR article.title IN $doc_ids\n'
           '\n'
           '        //가변 경로를 사용하여 1~2단계 깊이의 조문 찾기\n'
           '        MATCH (article)-[r*1..2]->(related:Article)\n'
           '\n'
           '        //결과 수집 밀 가공\n'
           '        WITH article, related,r\n'
           '        WITH article,\n'
           '             related,\n'
           '             size(r) AS path_length,\n'
           '             [rel IN r | type(rel)] AS relationship_types\n'
           '\n'
           '        // 최종 결과 반환\n'
           '        RETURN article.id AS article_id,\n'
           '               article.title AS title,\n'
           '               article.content AS content,\n'
           '               COLLECT(DISTINCT {\n'
           '                   id: related.id, \n'
           '                   title: related.title, \

In [23]:
# 실행 테스트
result = kg_enhanced_law_rag("법정 근로시간 연장이 가능한 특별한 사정에 대해서 설명해주세요.")
print(result["result"])

근로기준법에 대한 법률 전문가로서 질문에 답변해 드리겠습니다.

질문하신 "법정 근로시간 연장이 가능한 특별한 사정"에 대하여 근로기준법 제9조에서 규정하고 있습니다.

**근로기준법 제9조(특별한 사정이 있는 경우의 근로시간 연장 신청 등)**는 "법 제53조제4항 본문에서 '특별한 사정'이란 다음 각 호의 어느 하나에 해당하는 경우를 말한다"고 명시하고 있습니다. 이는 법정 근로시간을 연장할 수 있는 예외적인 '특별한 사정'이 무엇인지 구체적으로 열거하고 있음을 의미합니다.

그러나 제공된 법률 조문 정보에는 **제9조에서 언급하는 '다음 각 호'의 구체적인 내용이 명시되어 있지 않습니다.** 또한, 제9조가 인용하고 있는 **법 제53조제4항 본문의 내용 또한 제공되지 않아** 해당 '특별한 사정'이 무엇인지 정확하게 설명해 드리기 어렵습니다.

지식 그래프 정보에서도 제9조와 직접 또는 간접적으로 관련된 조문(제10조, 제11조)들이 감시 또는 단속적 근로자의 적용 제외 승인이나 15세 미만자의 취직인허 신청 등에 관한 내용으로, '특별한 사정'의 구체적인 내용과는 직접적인 연관성을 보여주지 않습니다.

따라서, 현재 제공된 정보만으로는 법정 근로시간 연장이 가능한 '특별한 사정'이 구체적으로 무엇인지 답변해 드릴 수 없습니다.


In [24]:
pprint(result['intermediate_steps'])

[{'query': '\n'
           '        //벡터 검색 결과와 일치하는 법률 조문 찾기\n'
           '        MATCH (article:Article)\n'
           '        WHERE article.id in $doc_ids OR article.title IN $doc_ids\n'
           '\n'
           '        //가변 경로를 사용하여 1~2단계 깊이의 조문 찾기\n'
           '        MATCH (article)-[r*1..2]->(related:Article)\n'
           '\n'
           '        //결과 수집 밀 가공\n'
           '        WITH article, related,r\n'
           '        WITH article,\n'
           '             related,\n'
           '             size(r) AS path_length,\n'
           '             [rel IN r | type(rel)] AS relationship_types\n'
           '\n'
           '        // 최종 결과 반환\n'
           '        RETURN article.id AS article_id,\n'
           '               article.title AS title,\n'
           '               article.content AS content,\n'
           '               COLLECT(DISTINCT {\n'
           '                   id: related.id, \n'
           '                   title: related.title, \

### 4.3 전문 검색(fulltext) 결합


#### 1) **Nori 분석기를 위한 인덱스 설정**

- 한국어 텍스트를 위한 Nori 분석기를 사용

In [26]:
cypher_query = """
// 한국어 법률 조문을 위한 전체 텍스트 인덱스 생성
CREATE FULLTEXT INDEX article_fulltext IF NOT EXISTS
FOR (a:Article) ON EACH [a.title, a.content]
OPTIONS {
  indexConfig: {
    `fulltext.analyzer`: 'nori',  // 한국어 분석기
    `fulltext.eventually_consistent`: true  // 성능 향상을 위한 설정
  }
}
"""

graph.query(cypher_query)

[#EA1D]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))): ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None)
Transaction failed and will be retried in 1.055496582697419s (Failed to write data to connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))))


[]

In [27]:
# Neo4j의 전문 검색 활용
fulltext_query = """
CALL db.index.fulltext.queryNodes("article_fulltext", $query) 
YIELD node, score
RETURN node.id AS id, node.title AS title, node.content AS content, score
ORDER BY score DESC
LIMIT 3
"""
question = "법정 근로시간 연장이 가능한 특별한 사정에 대해서 설명해주세요."
fulltext_results = graph.query(fulltext_query, {"query": question})

# 결과 출력
for record in fulltext_results:
    print(f"ID: {record['id']}")
    print(f"제목: {record['title']}")
    print(f"내용: {record['content']}")
    print(f"점수: {record['score']}")
    print("-" * 50)

ID: article_근로기준법 시행규칙(고용노동부령)(제00436호)(20250223)_제9조(특별한 사정이 있는 경우의 근로시간 연장 신청 등)
제목: 제9조(특별한 사정이 있는 경우의 근로시간 연장 신청 등)
내용: ① 법 제53조제4항 본문에서 '특별한 사정'이란 다음 각 호 의 어느 하나에 해당하는 경우를 말한다. &lt;신설 2020. 1. 31., 2021. 4. 5., 2024. 1. 5.&gt;
점수: 9.67167854309082
--------------------------------------------------
ID: article_근로기준법(법률)(제20520호)(20250223)_제53조(연장 근로의 제한)
제목: 제53조(연장 근로의 제한)
내용: ① 당사자 간에 합의하면 1주 간에 12시간을 한도로 제50조의 근로시간을 연장할 수 있다.
점수: 7.743139743804932
--------------------------------------------------
ID: article_근로기준법(법률)(제20520호)(20250223)_제59조(근로시간 및 휴게시간의 특례)
제목: 제59조(근로시간 및 휴게시간의 특례)
내용: ① 「통계법」 제22조제1항에 따라 통계청장이 고시하는 산업에 관한 표준의 중분 류 또는 소분류 중 다음 각 호의 어느 하나에 해당하는 사업에 대하여 사용자가 근로자대표와 서면으로 합의한 경우 에는 제53조제1항에 따른 주(週) 12시간을 초과하여 연장근로를 하게 하거나 제54조에 따른 휴게시간을 변경할 수 있다.
점수: 6.657968997955322
--------------------------------------------------


#### 2) **전문 검색과 벡터 검색을 결합한 Hybrid RAG**

In [28]:
def hybrid_kg_enhanced_law_rag(question):
    """전문 검색, 벡터 검색, 지식 그래프를 결합한 법률 RAG 시스템"""
    try:
        # 1. 벡터 검색으로 관련 법률 문서 찾기
        vector_docs = vector_store.similarity_search(question, k=3)
        
        # 검색된 문서의 ID 추출
        doc_ids = []
        for doc in vector_docs:
            if "id" in doc.metadata:
                doc_ids.append(doc.metadata["id"])
            elif "title" in doc.metadata:
                doc_ids.append(doc.metadata["title"])
                
        # 2. 전문 검색으로 관련 법률 문서 찾기
        # Neo4j의 전문 검색 활용
        fulltext_query = """
        CALL db.index.fulltext.queryNodes("article_fulltext", $query) 
        YIELD node, score
        RETURN node.id AS id, node.title AS title, node.content AS content, score
        ORDER BY score DESC
        LIMIT 3
        """
        
        fulltext_results = graph.query(fulltext_query, {"query": question})
        
        # 전문 검색 결과의 ID 추가
        for record in fulltext_results:
            if record["id"] and record["id"] not in doc_ids:
                doc_ids.append(record["id"])
                
        # 검색 결과가 없는 경우 처리
        if not doc_ids:
            return {
                "query": question,
                "result": "관련 법률 조항을 찾을 수 없습니다. 다른 질문을 시도해보세요.",
                "intermediate_steps": []
            }
        
        # 3. 그래프 검색: 가변 경로를 사용하여 관련 조문과 그 전후 조문 찾기 (2단계 깊이까지)
        cypher_query = """
        // 검색 결과와 일치하는 법률 조문 찾기
        MATCH (article:Article)
        WHERE article.id IN $doc_ids OR article.title IN $doc_ids
        
        // 가변 경로를 사용하여 1~2단계 깊이의 조문 찾기 (양방향 고려)
        MATCH path = (article)-[r*1..2]-(related:Article)
        WHERE article <> related  // 자기 자신과의 관계 제외
        
        // 결과 수집 및 가공
        WITH article, related, r, path
        WITH article, 
             related,
             size(r) AS path_length,
             [rel IN r | type(rel)] AS relationship_types
        
        // 최종 결과 반환
        RETURN article.id AS article_id, 
               article.title AS title,
               article.content AS content,
               COLLECT(DISTINCT {
                   id: related.id, 
                   title: related.title, 
                   content: related.content, 
                   path_length: path_length,
                   relationships: relationship_types
               }) AS relatedArticles
        """
        
        # 그래프 검색 실행 및 결과 처리
        graph_results = graph.query(cypher_query, {"doc_ids": doc_ids})
        
        # 4. 그래프 정보를 텍스트로 변환
        kg_context = ""
        for record in graph_results:
            kg_context += f"조문: {record['title']}\n내용: {record['content']}\n\n"
            
            # 관련 조문 정보 추가 (경로 길이에 따라 구분)
            kg_context += "관련 조문:\n"
            
            # 1단계와 2단계 관계를 구분하여 표시
            level1_articles = [rel for rel in record['relatedArticles'] if rel['path_length'] == 1]
            level2_articles = [rel for rel in record['relatedArticles'] if rel['path_length'] == 2]
            
            # 1단계 관련 조문 표시
            kg_context += "직접 관련 조문:\n"
            for article in level1_articles:
                rel_type = article['relationships'][0] if article['relationships'] else "관계 없음"
                kg_context += f"- {article['title']} (관계: {rel_type})\n  {article['content']}\n\n"
            
            # 2단계 관련 조문 표시
            kg_context += "간접 관련 조문:\n"
            for article in level2_articles:
                rel_types = " -> ".join(article['relationships']) if article['relationships'] else "관계 없음"
                kg_context += f"- {article['title']} (관계 경로: {rel_types})\n  {article['content']}\n\n"
        
        # 5. 원본 문서 내용 가져오기
        doc_context = "\n".join([f"{doc.metadata.get('title', '제목 없음')}: {doc.page_content}" for doc in vector_docs])
        
        # 6. 통합 컨텍스트 생성
        combined_context = f"법률 문서 정보:\n{doc_context}\n\n법률 지식 그래프 정보:\n{kg_context}"
        
        # 7. 프롬프트 템플릿 정의
        kg_template = """
        당신은 근로기준법에 대한 전문 지식을 갖춘 법률 전문가입니다.
        제공된 법률 조문과 지식 그래프 정보를 바탕으로 질문에 정확하게 답변해 주세요.
        
        지식 그래프는 법률 조문 간의 관계와 연결성을 보여줍니다.
        이 관계 정보를 활용하여 더 정확하고 포괄적인 법률 해석을 제공하세요.
        
        법률 조문이나 지식 그래프에서 찾을 수 없는 정보에 대해서는 솔직하게 모른다고 답변하세요.
        답변은 법률 용어를 적절히 사용하되, 일반인도 이해할 수 있도록 명확하게 작성해 주세요.
        
        참고할 정보:
        {context}
        
        질문: {question}
        
        답변:
        """
        
        kg_prompt = PromptTemplate(
            template=kg_template,
            input_variables=["context", "question"]
        )
        
        # 8. RAG 체인 구성 및 실행
        rag_chain = kg_prompt | llm | StrOutputParser()
        result = rag_chain.invoke({
            "question": question, 
            "context": combined_context
        })
        
        return {
            "query": question,
            "result": result,
            "intermediate_steps": [
                {"vector_search": [doc.metadata for doc in vector_docs]},
                {"fulltext_search": [dict(record) for record in fulltext_results]},
                {"graph_search": [dict(record) for record in graph_results]}
            ]
        }
    
    except Exception as e:
        return {
            "query": question,
            "result": f"검색 중 오류가 발생했습니다: {str(e)}",
            "error": str(e)
        }
    

# 실행 테스트
result = hybrid_kg_enhanced_law_rag("법정 근로시간 연장이 가능한 특별한 사정에 대해서 설명해주세요.")
print(result["result"])

[#F7A8]  _: <CONNECTION> error: Failed to write data to connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))): ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None)
Transaction failed and will be retried in 0.8305240507041798s (Failed to write data to connection IPv4Address(('localhost', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687))))


근로기준법에 대한 법률 전문가로서 질문에 답변드리겠습니다.

질문하신 "법정 근로시간 연장이 가능한 특별한 사정"에 대하여 근로기준법 제9조는 "법 제53조제4항 본문에서 '특별한 사정'이란 다음 각 호의 어느 하나에 해당하는 경우를 말한다"고 규정하고 있습니다.

그러나 제공된 법률 조문 정보와 지식 그래프 정보에는 근로기준법 제9조에서 언급하는 '다음 각 호'의 구체적인 내용, 즉 '특별한 사정'이 무엇인지에 대한 상세한 정보가 포함되어 있지 않습니다. 따라서 현재 주어진 정보만으로는 법정 근로시간 연장이 가능한 구체적인 '특별한 사정'이 무엇인지 설명해 드리기 어렵습니다.

다만, 참고로 근로기준법은 다음과 같은 연장 근로 관련 규정을 두고 있습니다:

*   **일반적인 연장 근로의 제한 (제53조제1항):** 당사자 간에 합의하면 1주 간에 12시간을 한도로 근로시간을 연장할 수 있습니다. 이는 일반적인 연장 근로의 한도를 정한 규정입니다.
*   **근로시간 및 휴게시간의 특례 (제59조제1항):** 「통계법」에 따라 통계청장이 고시하는 특정 산업(예: 운수업, 의료 및 보건업 등)에 해당하는 사업의 경우, 사용자가 근로자대표와 서면으로 합의하면 제53조제1항에 따른 주 12시간을 초과하여 연장근로를 하게 하거나 휴게시간을 변경할 수 있습니다. 이는 '특별한 사정'과는 별개로 특정 사업에 대한 '특례' 규정입니다.

결론적으로, 근로기준법 제53조제4항 본문에서 언급하는 '특별한 사정'의 구체적인 내용은 제공된 정보에서 확인할 수 없음을 알려드립니다.


In [29]:
pprint(result['intermediate_steps'])

[{'vector_search': [{'full_text': '제9조(특별한 사정이 있는 경우의 근로시간 연장 신청 등) ① 법 '
                                  "제53조제4항 본문에서 '특별한 사정'이란 다음 각 호 의 어느 하나에 "
                                  '해당하는 경우를 말한다. &lt;신설 2020. 1. 31., 2021. 4. '
                                  '5., 2024. 1. 5.&gt;',
                     'number': '9',
                     'title': '제9조(특별한 사정이 있는 경우의 근로시간 연장 신청 등)'},
                    {'full_text': '제53조(연장 근로의 제한) ① 당사자 간에 합의하면 1주 간에 12시간을 '
                                  '한도로 제50조의 근로시간을 연장할 수 있다.',
                     'number': '53',
                     'title': '제53조(연장 근로의 제한)'},
                    {'full_text': '제59조(근로시간 및 휴게시간의 특례) ① 「통계법」 제22조제1항에 따라 '
                                  '통계청장이 고시하는 산업에 관한 표준의 중분 류 또는 소분류 중 다음 각 호의 '
                                  '어느 하나에 해당하는 사업에 대하여 사용자가 근로자대표와 서면으로 합의한 경우 '
                                  '에는 제53조제1항에 따른 주(週) 12시간을 초과하여 연장근로를 하게 하거나 '
                                  '제54조에 따른 휴게시간을 변경할 수 있